# LLM-to-Database Strategy — RouteMind

**Scenario:** RouteMind, Portside Logistics' internal AI copilot, sits between dispatchers and two very different kinds of knowledge: structured operational data (shipments, drivers, freight cost) sitting in SQLite, and unstructured policy text (the driver handbook, the SLA policy, hazmat procedures) that will eventually live in a vector store. This notebook covers the basics of connecting an LLM workflow to both:

- Text-to-SQL with a local SQLite file
- When to use SQL, vector search, or hybrid retrieval
- Structured output into database rows
- LLM as a simple query planner / intent router

LangChain's SQL agent tutorial shows the SQL workflow for querying databases and notes the risk of model-generated SQL, so permissions should be narrowly scoped. Retrieval docs explain that LLMs have finite context and static knowledge, and retrieval helps by fetching relevant external knowledge at query time. Vector store docs describe vector stores as embedded data structures used for similarity search. Structured output docs show how LangChain can return predictable JSON or Pydantic-shaped data.

## Learning goals

By the end of this notebook, you should be able to:

1. Understand the basic idea behind text-to-SQL.
2. Decide when SQL is the right store, when vector search is the right store, and when both are useful.
3. Parse model output into a clean database row.
4. Build a small intent router that sends a dispatcher's prompt to the right data source.

## 1) Install dependencies

This notebook keeps the tooling simple.

```bash
pip install -U sqlmodel aiosqlite langchain langchain-groq pydantic python-dotenv
```

You can use either raw `sqlite3` or SQLModel. For the very first version, raw SQLite is enough.

In [ ]:
%pip install -qU sqlmodel aiosqlite langchain langchain-groq pydantic python-dotenv

## 2) The three basic data paths

RouteMind has to decide, for every dispatcher question, which store actually holds the answer.

### Use SQL when:
- the data is structured — shipment status, freight cost, driver assignments
- the fields are known in advance
- you need filters, counts, joins, or exact lookups

### Use vector search when:
- the information is mostly text — the driver handbook, the SLA policy, hazmat procedures
- the meaning matters more than exact fields
- you want semantic similarity search

### Use hybrid retrieval when:
- part of the answer is structured (which shipments are late)
- part of the answer is semantic text (what the SLA says about compensation for a late shipment)
- you want SQL filters plus vector search together

LangChain's retrieval docs say retrieval helps when LLMs cannot hold all content in context, and vector store docs describe similarity search over embedded data.

## 3) A tiny SQLite database

We will create a very small local database for RouteMind's own operating data:
- prompt log
- token usage
- model latency

This is enough for a basic LLM-to-database workflow, before we bring in the real shipments/drivers schema in the text-to-SQL lab.

In [ ]:
import sqlite3
from pathlib import Path
from datetime import datetime

db_path = Path("routemind_strategy.db")

conn = sqlite3.connect(db_path)
cur = conn.cursor()

cur.execute('''
CREATE TABLE IF NOT EXISTS routemind_prompt_log (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_prompt TEXT NOT NULL,
    llm_response TEXT,
    created_at TEXT NOT NULL
)
''')

cur.execute('''
CREATE TABLE IF NOT EXISTS routemind_token_usage (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    request_name TEXT NOT NULL,
    input_tokens INTEGER DEFAULT 0,
    output_tokens INTEGER DEFAULT 0,
    total_tokens INTEGER DEFAULT 0,
    created_at TEXT NOT NULL
)
''')

cur.execute('''
CREATE TABLE IF NOT EXISTS routemind_latency_log (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    request_name TEXT NOT NULL,
    model_name TEXT NOT NULL,
    latency_ms REAL NOT NULL,
    created_at TEXT NOT NULL
)
''')

conn.commit()
conn.close()

print("Created database:", db_path.resolve())

## 4) Raw SQL text-to-SQL idea

A text-to-SQL flow usually looks like this:

1. A dispatcher asks a question.
2. The LLM identifies the target table and fields.
3. The LLM generates a SQL query.
4. The app runs the query on SQLite.
5. The app returns the result.

LangChain's SQL agent tutorial follows a similar pattern: inspect schemas, choose relevant tables, generate a query, double-check it, execute it, and answer from the results. It also warns that model-generated SQL should run with narrow permissions.

In [ ]:
# Simple, safe SQL example without any LLM call
conn = sqlite3.connect(db_path)
cur = conn.cursor()

cur.execute(
    "INSERT INTO routemind_prompt_log (user_prompt, llm_response, created_at) VALUES (?, ?, ?)",
    ("Show me shipments dispatched from Chennai this week",
     "Here are the shipments dispatched from the Chennai depot this week...",
     datetime.utcnow().isoformat())
)
conn.commit()

cur.execute("SELECT id, user_prompt, llm_response FROM routemind_prompt_log ORDER BY id DESC LIMIT 5")
rows = cur.fetchall()
print(rows)

conn.close()

## 5) Query planner basics

At a high level, an LLM can act as a query planner for RouteMind.

It does not need to be a full agent in this notebook. A simple planner only needs to decide:

- SQL
- vector search
- hybrid
- direct answer

That is enough for a first pass before you move into deeper RAG and agent workflows — which is exactly what the guided labs later in this module do.

In [ ]:
def choose_store(user_prompt: str) -> str:
    prompt = user_prompt.lower()

    sql_keywords = ["count", "total", "sum", "average", "latest", "how many", "by date", "shipment", "delivery", "freight cost"]
    vector_keywords = ["meaning", "similar", "policy", "handbook", "procedure", "semantic", "compliance"]
    hybrid_keywords = ["and", "together", "both", "plus"]

    if any(k in prompt for k in hybrid_keywords) and any(k in prompt for k in sql_keywords + vector_keywords):
        return "hybrid"
    if any(k in prompt for k in sql_keywords):
        return "sql"
    if any(k in prompt for k in vector_keywords):
        return "vector"
    return "direct"

examples = [
    "How many shipments were delayed yesterday?",
    "What is our policy on driver rest breaks?",
    "Show delayed shipments and the SLA compensation guidance together.",
    "Explain what a service tier is.",
]

for q in examples:
    print(q, "->", choose_store(q))

## 6) SQL vs vector vs hybrid

### SQL
Best for:
- shipment status and freight cost
- driver assignments
- delivery counts and dates
- structured operational metadata

### Vector search
Best for:
- the driver handbook
- the customer SLA policy
- hazmat handling procedures
- semantic policy questions

### Hybrid
Best for:
- structured filters plus semantic matching — "which shipments are late, and what does the SLA say about compensation"
- search over policy documents with metadata filters
- cases where both exact fields and meaning matter

LangChain's retrieval docs explain that existing systems can be used as a tool or queried and passed back as context, and the vector store docs show similarity search plus metadata filtering.

## 7) Structured output into database rows

A simple pattern is:

1. Ask the model for structured output.
2. Validate it with a Pydantic schema.
3. Insert the validated object into SQLite.

LangChain's structured output docs say the output can be returned as JSON objects, Pydantic models, or dataclasses, and the data is validated before use.

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class PromptRecord(BaseModel):
    user_prompt: str = Field(...)
    category: str = Field(...)
    answer: str = Field(...)

# Example of model output already shaped like structured data
raw_output = {
    "user_prompt": "How many shipments were delayed yesterday?",
    "category": "sql",
    "answer": "Use a SQL query against the shipments table, filtered by status and dispatch date.",
}

try:
    record = PromptRecord(**raw_output)
    print(record)
except ValidationError as e:
    print(e)

In [ ]:
# Insert the validated row into SQLite
conn = sqlite3.connect(db_path)
cur = conn.cursor()

record = PromptRecord(
    user_prompt="How many shipments were delayed yesterday?",
    category="sql",
    answer="Use a SQL query against the shipments table, filtered by status and dispatch date.",
)

cur.execute(
    "INSERT INTO routemind_prompt_log (user_prompt, llm_response, created_at) VALUES (?, ?, ?)",
    (record.user_prompt, record.answer, datetime.utcnow().isoformat())
)

conn.commit()
conn.close()

print("Inserted structured record into routemind_prompt_log")

## 8) Optional: LLM-backed structured output

If you later connect a model, keep the same schema and only swap the input source.

A simple pattern is:

- prompt the model for JSON
- parse into `PromptRecord`
- store in SQLite

That gives you a stable database row format even if the prompt text changes.

In [ ]:
# Groq-backed structured output. Falls back to an offline canned example when
# the Groq API isn't reachable -- e.g. in a sandbox with no network egress to
# api.groq.com -- so the rest of the pipeline still has a JSON string to
# validate against. On a machine with a real GROQ_API_KEY and network access,
# the live branch runs instead.
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import json

question = "How many delayed shipments did we have in Chennai last week?"

try:
    llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Return only JSON with keys: user_prompt, category, answer."),
        ("user", "{question}")
    ])
    chain = prompt | llm | StrOutputParser()
    raw_json = chain.invoke({"question": question})
    print("[live-groq]", raw_json)
except Exception as e:
    raw_json = json.dumps({
        "user_prompt": question,
        "category": "sql",
        "answer": "Query the shipments table filtered by origin_depot='Chennai' and status='delayed'.",
    })
    print(f"[offline-fallback: {type(e).__name__}]", raw_json)

print("LLM template cell ready.")

## 9) Hybrid routing logic

A planner can route to a store based on the dispatcher's intent.

This version stays intentionally simple:

- SQL for structured metrics
- vector for semantic policy text
- hybrid for mixed cases
- direct answer for general explanations

In [ ]:
def route_request(user_prompt: str) -> str:
    decision = choose_store(user_prompt)

    if decision == "sql":
        return "Route to SQLite (shipments / drivers / customers)"
    if decision == "vector":
        return "Route to vector search (driver handbook / SLA policy / hazmat procedures)"
    if decision == "hybrid":
        return "Route to SQL + vector search"
    return "Answer directly"

for q in examples:
    print(q, "->", route_request(q))

## 10) Why this matters

This is the small version of the larger architecture:

- the LLM decides where the data lives
- the app executes the right retrieval step
- the result is returned in a consistent structure

That is the core idea behind RouteMind's query planner, and it is exactly what the guided text-to-SQL lab builds on next.

## 11) Keep it safe and simple

For early implementations:

- use read-only SQL where possible
- keep the schema narrow
- validate structured outputs before insert
- use vector search only for text that needs semantic matching
- do not overbuild agents before the basic routing is working

LangChain's SQL tutorial explicitly notes that model-generated SQL has risks and permissions should be scoped narrowly.

One more caution specific to `choose_store` as written: it matches keywords as plain substrings, not whole words. That is a real gotcha, not a hypothetical one -- try `choose_store("What does the driver handbook say about rest breaks?")` and it comes back `"hybrid"` instead of `"vector"`, because the word **hand**book contains the substring **and**, which is one of the hybrid keywords. Naive substring routing like this is fine for a first pass, but it is exactly the kind of fragility that motivates moving to a real LLM-based intent classifier (or at minimum, whole-word matching) before this goes anywhere near production.

## 12) What comes next

This notebook is intentionally basic.

Later, you can expand it into:

- real text-to-SQL with LangChain SQL tools, over the real shipments/drivers schema
- a proper vector store for the driver handbook, SLA policy, and hazmat procedures
- hybrid retrieval pipelines
- agent-based query planning
- evaluation and tracing

For now, the goal is just to understand the decision boundary between the data stores.

## Key takeaways

- SQL is best for structured data and metrics -- shipments, drivers, freight cost.
- Vector search is best for semantic text matching -- policy and procedure documents.
- Hybrid retrieval is useful when both matter, like a late-shipment question that also needs the SLA wording.
- Structured output helps you write clean rows into SQLite.
- A simple query planner can route requests before you build deeper RAG or agent workflows -- but naive keyword matching is fragile, as the substring gotcha above shows.

## Verification status in this environment

| Component | Status |
|---|---|
| SQLite schema, inserts, `PromptRecord` structured-output validation | Executed and verified |
| `choose_store` / `route_request` keyword routing (all 4 categories) | Executed and verified |
| Groq-backed structured output (section 8) | Offline fallback executed -- this sandbox has no network route to `api.groq.com`. On a machine with a real `GROQ_API_KEY`, the live branch runs instead with no code changes needed. |

## References

- SQL agent: https://docs.langchain.com/oss/python/langchain/sql-agent
- Vector store integrations: https://docs.langchain.com/oss/python/integrations/vectorstores/index
- Retrieval: https://docs.langchain.com/oss/python/langchain/retrieval#building-a-knowledge-base
- Structured output: https://docs.langchain.com/oss/python/langchain/structured-output